In [ ]:
# Example 1: Public API surface — primary entry points exist and are callable.
import httpx_ws

expected_names = [
    'connect_ws',
    'aconnect_ws',
    'WebSocketSession',
    'AsyncWebSocketSession',
    'WebSocketDisconnect',
    'WebSocketNetworkError',
    'WebSocketUpgradeError',
    'WebSocketInvalidTypeReceived',
]

present = {n: hasattr(httpx_ws, n) for n in expected_names}
for name, ok in present.items():
    obj = getattr(httpx_ws, name, None)
    print(f"  {name}: {'OK' if ok else 'MISSING'} -> {type(obj).__name__ if obj is not None else 'N/A'}")

assert all(present.values()), f"Missing public API names: {[n for n, ok in present.items() if not ok]}"

In [ ]:
# Example 2: Exception class hierarchy — errors are proper Exception subclasses.
from httpx_ws import (
    WebSocketDisconnect,
    WebSocketInvalidTypeReceived,
    WebSocketNetworkError,
    WebSocketUpgradeError,
)

for exc_cls in (WebSocketDisconnect, WebSocketInvalidTypeReceived, WebSocketNetworkError, WebSocketUpgradeError):
    assert issubclass(exc_cls, Exception), f"{exc_cls.__name__} is not an Exception"
    print(f"  {exc_cls.__name__}: MRO depth = {len(exc_cls.__mro__)}")

# WebSocketDisconnect carries close code + reason.
try:
    raise WebSocketDisconnect(code=1000, reason="normal closure")
except WebSocketDisconnect as e:
    print(f"  raised disconnect: code={e.code}, reason={e.reason!r}")
    assert e.code == 1000
    assert e.reason == "normal closure"

In [ ]:
# Example 3: connect_ws / aconnect_ws are context-manager factories with the expected signatures.
import inspect

from httpx_ws import aconnect_ws, connect_ws

sync_sig = inspect.signature(connect_ws)
async_sig = inspect.signature(aconnect_ws)

print(f"connect_ws{sync_sig}")
print(f"aconnect_ws{async_sig}")

# Both should accept a URL as the first positional argument.
sync_params = list(sync_sig.parameters)
async_params = list(async_sig.parameters)
assert 'url' in sync_params, f"connect_ws missing 'url': {sync_params}"
assert 'url' in async_params, f"aconnect_ws missing 'url': {async_params}"
print("  both accept url as a parameter")

In [ ]:
# Example 4: WebSocketSession / AsyncWebSocketSession expose the expected message methods.
from httpx_ws import AsyncWebSocketSession, WebSocketSession

sync_methods = ['send_text', 'send_bytes', 'send_json', 'receive_text', 'receive_bytes', 'receive_json', 'ping', 'close']
async_methods = ['send_text', 'send_bytes', 'send_json', 'receive_text', 'receive_bytes', 'receive_json', 'ping', 'close']

for method in sync_methods:
    assert hasattr(WebSocketSession, method), f"WebSocketSession missing {method}"
for method in async_methods:
    assert hasattr(AsyncWebSocketSession, method), f"AsyncWebSocketSession missing {method}"

print(f"  WebSocketSession methods OK: {sync_methods}")
print(f"  AsyncWebSocketSession methods OK: {async_methods}")

In [ ]:
# Example 5: Underlying deps (httpx + wsproto) are wired correctly — frame encoding round-trip.
import wsproto
import wsproto.events

import httpx_ws

# wsproto provides the WebSocket framing layer that httpx_ws builds on.
client_conn = wsproto.WSConnection(wsproto.ConnectionType.CLIENT)
server_conn = wsproto.WSConnection(wsproto.ConnectionType.SERVER)

# Drive the protocol enough to verify the dep graph is healthy.
data = client_conn.send(wsproto.events.Request(host='example.com', target='/'))
assert b'Upgrade: websocket' in data, "wsproto did not produce a websocket upgrade request"
server_conn.receive_data(data)
events = list(server_conn.events())
assert any(isinstance(e, wsproto.events.Request) for e in events), "server did not parse upgrade request"

print(f"  httpx_ws imported from: {httpx_ws.__file__ if hasattr(httpx_ws, '__file__') else httpx_ws.__path__}")
print(f"  wsproto version: {wsproto.__version__}")
print(f"  upgrade-request bytes: {len(data)}")
assert len(data) > 0